# KagMCP — MCP no oficial para Kaggle

**KagMCP** es un servidor no oficial de Model Context Protocol que permite conectar ChatGPT u otros agentes compatibles a una sesión autorizada de Kaggle para análisis de datos, investigación, depuración de notebooks y desarrollo de machine learning.

Integra **MCP sobre HTTPS con OAuth y PIN**, herramientas de archivos/terminal, **GitHub con token personal opcional** y acceso **SSH/SFTP opcional** a través de un VPS. Todo se ejecuta dentro de un **único binario Go**. No está afiliado a Kaggle, Google ni OpenAI.

**Requisito:** habilita Internet en el notebook y configura al menos los Secrets `MCP_ENABLED=true` y `MCP_ACCESS_PIN` (12 caracteres o más).

## 1. Arquitectura

```text
ChatGPT / agente MCP -> HTTPS trycloudflare.com -> Kaggle:127.0.0.1:8181/mcp
                                                    |
                                                 KagMCP Go
                                                    |-- archivos en /kaggle/working
                                                    |-- datasets en /kaggle/input (solo lectura)
                                                    |-- procesos/comandos del runtime
                                                    |-- GitHub REST con GITHUB_TOKEN (opcional)
                                                    \-- SSH/SFTP por VPS (opcional)
```

El enlace HTTPS temporal se genera automáticamente cuando `MCP_TUNNEL=cloudflare` (valor predeterminado). La URL aparecerá en los logs de la celda de arranque. **No necesitas VPS para MCP**.

**Importante:** `/kaggle/working/.kagmcp` es el directorio privado de estado OAuth, caché de Cloudflare y configuración futura. Kaggle puede descartar los archivos en un runtime nuevo; no supongas persistencia garantizada.

Si activas SSH con `SSH_ENABLED=true`, KagMCP conserva el túnel externo del VPS a `0.0.0.0` y las credenciales SSH originales.

## 2. Clonar el repositorio remoto y compilar en Kaggle

No necesitas subir un ejecutable. **Ejecuta las siguientes tres celdas en orden** con Internet habilitado: clonar/actualizar `main` desde GitHub (sin sobrescribir cambios locales), comprobar/descargar Go y compilar el binario directamente en `/kaggle/working`. Si Go no está instalado o es anterior al mínimo de `go.mod`, se descarga de **go.dev** con comprobación del SHA-256 publicado por el equipo de Go. El binario se reconstruye cada vez que ejecutas la celda de compilación.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import tempfile
import urllib.request

REPO_URL = "https://github.com/ThowiLabs/kagssh-go.git"
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "kagssh-go"
BINARY = WORKING / "kagmcp-linux-amd64"

if platform.system() != "Linux" or platform.machine().lower() not in ("x86_64", "amd64"):
    raise RuntimeError("Este ejemplo requiere Kaggle Linux x86_64 (amd64).")
if not WORKING.is_dir():
    raise RuntimeError("Ejecuta este notebook en Kaggle (/kaggle/working).")
if not shutil.which("git"):
    raise RuntimeError("No se encontró git en el entorno de Kaggle.")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True, timeout=180,
    )
else:
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} existe pero no es un clon Git. Muévelo antes de continuar.")
    current_origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip().rstrip("/")
    if current_origin.removesuffix(".git").lower() != REPO_URL.removesuffix(".git").lower():
        raise RuntimeError(f"El directorio ya existe y su remoto es distinto: {current_origin}")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("El clon tiene cambios sin guardar; no se actualizará ni se borrará nada.")
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main"],
        check=True, timeout=180,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "merge", "--ff-only", "origin/main"],
        check=True, timeout=60,
    )

REVISION = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print("Repositorio listo:", REPO_DIR)
print("Revisión remota clonada:", REVISION)

In [ ]:
# Usa el Go del sistema si cumple el mínimo de go.mod; si falta o es antiguo,
# instala la versión necesaria desde la distribución oficial con SHA-256 verificado.
mod_text = (REPO_DIR / "go.mod").read_text(encoding="utf-8")
match = re.search(r"(?m)^go\s+(\d+)\.(\d+)(?:\.(\d+))?\s*$", mod_text)
if not match:
    raise RuntimeError("No se encontró la versión mínima de Go en go.mod.")
MIN_GO = tuple(int(v or 0) for v in match.groups())
required = ".".join(str(v) for v in MIN_GO)
go_executable = shutil.which("go")

def installed_version(executable):
    if not executable:
        return None
    result = subprocess.run([executable, "version"], capture_output=True, text=True)
    found = re.search(r"go(\d+)\.(\d+)\.(\d+)", result.stdout)
    return tuple(map(int, found.groups())) if result.returncode == 0 and found else None

existing = installed_version(go_executable)
# Go 1.27.0 y 1.27.1 preceden a los parches del 8/oct/2026.
def secure_go(version):
    return (version is not None and version >= MIN_GO
            and (version[:2] != (1, 27) or version[2] >= 2))

if secure_go(existing):
    GO_BIN = Path(go_executable)
    print("Go instalado suficiente:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
else:
    if not os.environ.get("KAGGLE_USER_SECRETS_TOKEN") and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        print("Nota: esta instalación necesita acceso a Internet desde Kaggle.")
    release_version = "go" + required
    catalog_url = "https://go.dev/dl/?mode=json&include=all"
    with urllib.request.urlopen(catalog_url, timeout=40) as response:
        releases = json.load(response)
    release = next((item for item in releases if item.get("version") == release_version), None)
    if release is None:
        raise RuntimeError(f"La versión {release_version} no figura en las descargas oficiales.")
    archive_meta = next(
        (f for f in release["files"] if f.get("os") == "linux"
         and f.get("arch") == "amd64" and f.get("kind") == "archive"), None
    )
    if archive_meta is None or not re.fullmatch(r"[0-9a-f]{64}", archive_meta.get("sha256", "")):
        raise RuntimeError("No se pudo obtener el SHA-256 oficial de Go linux-amd64.")
    tempdir = Path(tempfile.mkdtemp(prefix="kagssh-toolchain-", dir="/tmp"))
    archive = tempdir / archive_meta["filename"]
    address = "https://go.dev/dl/" + archive_meta["filename"]
    print("Descargando Go verificado:", archive_meta["filename"])
    digest = hashlib.sha256()
    with urllib.request.urlopen(address, timeout=120) as source, archive.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != archive_meta["sha256"]:
        archive.unlink(missing_ok=True)
        raise RuntimeError("SHA-256 inválido: se rechaza la instalación.")
    subprocess.run(["tar", "-xzf", str(archive), "-C", str(tempdir)], check=True, timeout=120)
    archive.unlink(missing_ok=True)
    GO_BIN = tempdir / "go" / "bin" / "go"
    if not GO_BIN.is_file() or installed_version(str(GO_BIN)) != MIN_GO:
        raise RuntimeError("No se pudo validar la versión instalada de Go.")

print("Go seleccionado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())
print("Versión mínima del repositorio:", required)

In [ ]:
# Compila el código clonado para el runtime Linux de Kaggle (sin CGO).
build_env = os.environ.copy()
build_env.update({"CGO_ENABLED": "0", "GOOS": "linux", "GOARCH": "amd64", "GOTOOLCHAIN": "local"})
subprocess.run(
    [str(GO_BIN), "build", "-buildvcs=false", "-trimpath", "-ldflags=-s -w",
     "-o", str(BINARY), "./cmd/kagssh"],
    cwd=str(REPO_DIR), env=build_env, check=True, timeout=600,
)
version = subprocess.run([str(BINARY), "-version"], capture_output=True, text=True, check=True)
print("Compilación terminada:", BINARY)
print("Versión binaria:", version.stdout.strip())
print("Go usado:", subprocess.check_output([str(GO_BIN), "version"], text=True).strip())

## 3. Kaggle Secrets — sin crear una GitHub App

En **Add-ons → Secrets**, crea y autoriza las etiquetas que necesites. El binario lee los Secrets directamente; la prioridad es **variable explícita → Kaggle Secret → preferencia no secreta en `/kaggle/working/.kagmcp/config.json` → valor predeterminado**. No escribas tokens o PIN en el notebook.

| Etiqueta | Valor | Uso |
|---|---|---|
| `MCP_ENABLED` | `true` | Habilita el servidor MCP |
| `MCP_ACCESS_PIN` | PIN secreto de 12–128 caracteres | Protege autorización OAuth de ChatGPT |
| `MCP_TUNNEL` | `cloudflare` (default) | URL HTTPS temporal automática |
| `GITHUB_TOKEN` | Token personal fine-grained/PAT (opcional) | Conexión GitHub por API; **no se crea ninguna App** |
| `MCP_LISTEN_PORT` | `8181` (default) | Puerto HTTP local del MCP |
| `SSH_ENABLED` | `false` para solo MCP | Activa SSH/SFTP si tienes VPS |

Si necesitas **SSH**, configura además los Secrets `SSH_HOST`, `SSH_PASSWORD` (o `SSH_KEY`) y `SSH_LOGIN_PASSWORD` (o `SSH_AUTHORIZED_KEYS`). `SSH_USER` predetermina `root`, `SSH_PORT_REMOTE` a 22 y `SSH_PORT_KAGGLE` a 2223. SSH es opcional; puedes activar ambos servicios al mismo tiempo.

**GitHub:** un token fine-grained debe tener acceso a los repositorios seleccionados, `Contents` de lectura para consultar y escritura para guardar cambios. KagMCP no utiliza GitHub Apps ni solicita client secret.

**Seguridad:** el PIN de MCP sirve para aprobar clientes en OAuth; es independiente de `GITHUB_TOKEN`. `GITHUB_TOKEN` nunca se debe exponer al cliente como texto ni guardar en `.kagmcp`.

Para una URL fija con un proxy HTTPS propio, usa `MCP_TUNNEL=none` y `MCP_PUBLIC_URL=https://tu-dominio-publico`, apuntando el proxy al puerto HTTP local. `localhost.run` aún no se ha integrado a este binario.

## 4. Validar Secrets

La celda siguiente ejecuta `-check` y verifica los Secrets y parámetros sin abrir todavía los túneles. Configura `MCP_ENABLED` y `MCP_ACCESS_PIN` antes de continuar.

In [ ]:
# Comprueba que el binario puede leer la configuración, sin abrir el túnel.
check = subprocess.run(
    [str(BINARY), "-check"],
    text=True,
    capture_output=True,
    timeout=180,
)
if check.returncode != 0:
    # El programa no debería incluir los valores de Secrets en sus errores.
    # Revisa la configuración y evita compartir capturas/logs privados.
    raise RuntimeError(
        "Configuración inválida (o error al consultar Kaggle Secrets). "
        "Revisa etiquetas, autorización, Internet y huella del VPS. "
        f"Detalle: {check.stderr.strip()[:600]}"
    )
print(check.stdout.strip())
print("La validación NO garantiza que el VPS acepte el túnel.")

## 5. Iniciar KagMCP y mostrar logs en vivo

La siguiente celda permanecerá **ejecutándose normalmente**. Verás en su salida la URL `https://...trycloudflare.com/mcp` cuando el túnel esté listo.

Puedes conectar ChatGPT agregando esa URL como un servidor MCP remoto y aprobar el acceso en la pantalla OAuth utilizando tu `MCP_ACCESS_PIN`.

**Para detenerlo**, pulsa **Detener/Interruptar** en la celda de Kaggle; eso termina el proceso Go, el túnel HTTPS y el SSH opcional. No utilizamos una ejecución Python en segundo plano ni un visor de logs.

In [ ]:
# Mantiene la celda de Kaggle en ejecución y muestra los logs de Go directamente.
# Usa Detener / Interruptar celda en Kaggle para cerrar KagMCP.
import subprocess

print("Iniciando KagMCP. Pulsa Detener/Interruptar en Kaggle para desconectar.", flush=True)
with subprocess.Popen(
    [str(BINARY)],
    cwd=str(WORKING),
    stdin=subprocess.DEVNULL,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    errors="replace",
    # Se reenvía cada línea a print() de Jupyter, sin hilos ni segundo plano.
) as kagssh:
    try:
        for line in kagssh.stdout:  # Bucle bloqueante: mantiene activa la celda.
            print(line, end="", flush=True)  # Logs visibles en la misma celda.
        status = kagssh.wait()
    except KeyboardInterrupt:
        print("\nInterrupción de Kaggle: cerrando KagMCP...", flush=True)
        if kagssh.poll() is None:
            kagssh.terminate()
        try:
            kagssh.wait(timeout=10)
        except subprocess.TimeoutExpired:
            kagssh.kill()
            kagssh.wait()
        print("KagMCP detenido; túnel cerrado.", flush=True)
    else:
        if status != 0:
            raise RuntimeError(f"KagMCP terminó con error (código {status}). Revisa los logs anteriores.")
        print("KagMCP terminó normalmente.", flush=True)

## 6. Conectar ChatGPT o un cliente MCP

Cuando aparezca en los logs:

```text
KagMCP listo para conectar cliente MCP url=https://...trycloudflare.com/mcp
```

Agrega **esa URL HTTPS completa** a la configuración del cliente MCP compatible. Sigue el flujo de autenticación OAuth y escribe el PIN de `MCP_ACCESS_PIN` cuando se solicite. El servidor publica metadata de descubrimiento OAuth y responde a solicitudes MCP en `/mcp`.

El servidor está diseñado para clientes MCP que admiten herramientas y OAuth. Ni ChatGPT ni los agentes reciben acceso automáticamente: tienes que conectar el servidor y autorizarlo.

**SSH opcional:** si se activó `SSH_ENABLED` y tu VPS tiene `GatewayPorts clientspecified`, puedes seguir utilizando desde otra terminal:

```bash
ssh -p 2223 usuario_kaggle@IP_PUBLICA_DEL_VPS
sftp -P 2223 usuario_kaggle@IP_PUBLICA_DEL_VPS
```

El VPS debe permitir TCP 2223 y forwarding; si no activas SSH, no necesitas VPS.

## 7. Herramientas disponibles

- `environment_info`: descubre el entorno Kaggle.
- `list_dir`, `read_file`, `write_file`: inspecciona y edita archivos. `/kaggle/input` es solo lectura; `/kaggle/working` admite escritura.
- `exec`: ejecuta comandos con límites de tiempo y salida, trabajando desde `/kaggle/working`.
- `github_status`, `github_repositories`, `github_branches`: consulta GitHub mediante tu PAT.
- `github_file_read`, `github_file_write`: lee archivos o crea commits de archivos de GitHub mediante tu token.

El acceso MCP es **direct-host**: después de autorizar al agente, sus comandos corren con los permisos que tenga el proceso KagMCP. Debes confiar en el cliente antes de autorizarlo.

## Límites y seguridad

- Proyecto **independiente y no oficial**; respeta las políticas, cuotas y duración de los runtimes Kaggle.
- Los tokens y el PIN se leen de Kaggle Secrets, nunca de la URL ni de un archivo del repositorio. Los procesos lanzados por herramientas no reciben las variables sensibles.
- Protege el PIN; cualquier cliente que autorices puede ejecutar comandos y modificar archivos con los permisos del proceso.
- La carpeta `/kaggle/working/.kagmcp` conserva estado privado **durante el runtime**, pero Kaggle no garantiza persistencia entre sesiones nuevas.
- El servidor MCP requiere OAuth y rechaza acceso no autorizado. Cloudflare termina TLS y el origen HTTP permanece en `127.0.0.1`.
- `GITHUB_TOKEN` es un token **personal**, sin registrar una GitHub App; usa el menor alcance posible.
- La integración Cloudflare, autenticación y GitHub requiere una **prueba real dentro de Kaggle**. Las pruebas locales comprueban módulos y protocolo pero no garantizan que la red externa funcione.

Documentación: `README.md` y `USO_Y_CONFIGURACION.md` en el [repositorio provisional ThowiLabs/kagssh-go](https://github.com/ThowiLabs/kagssh-go).